# Train Deep Learning Model

This notebook consists of two tasks. For both tasks, you must provide your assigned dataset name in list format, just as you did in the previous exercise.

When you train a model, it will print the following evaluation metrics:
- `Training loss`
- `Training time`
- `Mean Absolute Error (MAE)`
- `Root Mean Squared Error (RMSE)`
- `R² score`

Below is an overview of the three metrics:

### Evaluation Metrics and Their Desired Trends

| Metric             | Definition                                                                                  | Desired Trend                                  |
|--------------------|----------------------------------------------------------------------------------------------|------------------------------------------------|
| MAE (Mean Absolute Error) | Measures the average absolute difference between predicted and actual values.           | ↓ Decrease — Lower MAE indicates better average accuracy. |
| RMSE (Root Mean Squared Error)  | Measures the average of squared differences; penalizes larger errors more heavily.      | ↓ Decrease — Lower RMSE indicates fewer large errors.       |
| R² (Coefficient of Determination) | Indicates the proportion of variance explained by the model (maximum = 1.0).        | ↑ Increase — Higher R² means better model fit.             |


Please familiarize yourself with the metrics above, as they will be important for completing the analysis in the next tutorial.

After model evaluation is complete, a folder will be created in your current directory using the following structure:
`dataset_name → model_name → activation_function → optimizer_name → epoch_num`

Three JSON files will be generated within this directory structure: 
1) `predictions.json`
2) `evaluation.json`
3) `train_loss.json`

The predictions.json file stores the input data points used for forecasting, along with the predicted values and corresponding target values. The evaluation.json file contains the evaluation metrics, while the train_loss.json file records the training loss and model training time.

## Task 1: Train Predefined Models

This task involves training four predefined model architectures using the dataset you have been assigned. These models include:

* `NN`
* `RNN` (Recurrent Neural Network)
* `LSTM` (Long Short-Term Memory)
* `GRU` (Gated Recurrent Unit)

Note: `LSTM` and `GRU` are specialized types of `RNNs` designed to handle sequence data more effectively.

You are not required to train all combinations — there are 125 or more available — but you should train at least 30 different combinations. Make sure these include at least four combinations for each model architecture (`NN`, `RNN`, `LSTM`, and `GRU`) to ensure broad coverage.

Example Model Combinations
```
| Model | Activation Function | Optimizer | Epochs |
|-------|---------------------|-----------|--------|
| NN    | ReLU                | AdamW     | 10     |
| RNN   | Tanh                | SGD       | 5      |
| LSTM  | GELU                | Adam      | 10     |
| GRU   | Leaky ReLU          | AdamW     | 15     |
```

The code is configured to run over 125 model combinations, which may take significant time to complete.

To speed up development and avoid long runtimes, you can reduce the number of combinations by limiting the range of options.

Shrinked Option Set (Example 1)
```
model_classes = ["NN", "RNN"]
activations = ["relu", "tanh"]
optimizers = ["adam"]
epoch_options = [5, 10]
```

Shrinked Option Set (Example 2 — Quick Debug)
```
model_classes = ["NN"]
activations = ["relu"]
optimizers = ["adam"]
epoch_options = [5]
```

Note: Do not attempt to train for more than 30 epochs per combination, as this may lead to long execution times and unnecessary resource usage.

## Task 2: Build and Train Your Custom Model

This task focuses on building and training your own custom model, referred to as `MyNN`.

In this task, you will design your own model architecture and evaluate its performance by training it under different settings. You should train at least `10` different combinations using `MyNN`.

Example Combinations with `MyNN`
```
| Model | Activation Function | Optimizer | Epochs |
|-------|---------------------|-----------|--------|
| MyNN  | ReLU                | AdamW     | 10     |
| MyNN  | Tanh                | SGD       | 5      |
| MyNN  | GELU                | Adam      | 10     |
| MyNN  | Leaky ReLU          | AdamW     | 15     |
```

In [1]:
# ------------------- #
# --- Do Not Edit --- #
# ------------------- #
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from buildings_bench import load_torch_dataset
from buildings_bench.models import model_factory

import tomli
from pathlib import Path
import argparse
import os 
import time
import json
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

class DataHandler:
    """
    Thin convenience wrapper around load_torch_dataset() + DataLoader
    construction, so Trainer doesn't need to know loading details.

    Usage:
        handler = DataHandler(batch_size=32)
        buildings = handler.load_dataset('ideal', scaler_transform='boxcox')
        loader = handler.create_dataloader(buildings[0][1])

    Args:
        batch_size (int): Batch size used by every DataLoader this creates.
    """
    def __init__(self, batch_size=32):
        self.batch_size = batch_size

    def load_dataset(self, dataset_name, scaler_transform):
        """Load a BuildingsBench dataset as a list of (building_id, dataset) pairs.

        Requires the TRANSFORM_PATH environment variable to already be set
        (path to the pickled scaler-transform data, e.g. for box-cox).

        Args:
            dataset_name (str): Dataset to load, e.g. 'ideal'.
            scaler_transform (str): '' | 'boxcox' | 'standard' -- which
                scaling transform to apply to the load values.

        Returns:
            list[tuple[str, TorchBuildingDataset]]: One entry per building.
        """
        from buildings_bench import load_torch_dataset
        return list(load_torch_dataset(
            dataset_name,
            apply_scaler_transform=scaler_transform,
            scaler_transform_path=Path(os.environ["TRANSFORM_PATH"])
        ))

    def create_dataloader(self, dataset):
        """Wrap a single building's dataset in a DataLoader.

        Args:
            dataset (TorchBuildingDataset): One building's windowed dataset.

        Returns:
            DataLoader: Batches of size self.batch_size. Not shuffled --
                order matters since these are time-series windows.
        """
        return DataLoader(dataset, batch_size=self.batch_size, shuffle=False)


class TimeSeriesSinusoidalPeriodicEmbedding(nn.Module):
    """
    Embeds a single periodic time feature (e.g. hour_of_day, which wraps
    23 -> 0) via sin/cos, then projects it to `embedding_dim`. Turns a
    value that wraps around into a smooth periodic representation a
    neural net can actually learn continuity from, instead of seeing a
    discontinuous jump at the wraparound point.

    Args:
        embedding_dim (int): Size of the output embedding.
    """
    def __init__(self, embedding_dim: int):
        super().__init__()
        self.linear = nn.Linear(2, embedding_dim)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Args:
            x (torch.Tensor): shape (batch, seq_len, 1) -- the raw periodic value.

        Returns:
            torch.Tensor: shape (batch, seq_len, embedding_dim).
        """
        x = torch.cat([torch.sin(torch.pi * x), torch.cos(torch.pi * x)], dim=2)
        return self.linear(x)


class Model(nn.Module):
    """
    Base class for every forecasting architecture in this script (NN, RNN,
    LSTM, GRU, MyNN). Handles the plumbing shared by all of them: fixed
    context/prediction window lengths, activation-function lookup by name,
    and the per-feature input embeddings (lat/lon, building type, load,
    and the three periodic time features). Subclasses only need to
    implement `_build_model()` and `forward()`.

    Attributes:
        context_len (int): Hours of history fed to the model (168 = 1 week).
        pred_len (int): Hours to forecast (24 = 1 day).
        activation (nn.Module): Resolved activation function.
        embeddings (nn.ModuleDict): Per-feature embedding layers.
    """
    DEFAULT_CONTEXT_LEN = 168
    DEFAULT_PRED_LEN = 24

    def __init__(self, activation):
        super().__init__()
        self.context_len = self.DEFAULT_CONTEXT_LEN
        self.pred_len = self.DEFAULT_PRED_LEN
        self.activation = self._get_activation(activation)
        self.embeddings = self._create_embeddings()

    def _create_embeddings(self):
        """Build the embedding layer for each input feature.

        Returns:
            nn.ModuleDict: keys 'power', 'building', 'lat', 'lon',
                'day_of_year', 'day_of_week', 'hour_of_day'.
        """
        return nn.ModuleDict({
            'power': nn.Linear(1, 64),
            'building': nn.Embedding(2, 32),
            'lat': nn.Linear(1, 32),
            'lon': nn.Linear(1, 32),
            'day_of_year': TimeSeriesSinusoidalPeriodicEmbedding(32),
            'day_of_week': TimeSeriesSinusoidalPeriodicEmbedding(32),
            'hour_of_day': TimeSeriesSinusoidalPeriodicEmbedding(32)
        })

    def _get_activation(self, name):
        """Look up an activation module by name (case-insensitive).

        Args:
            name (str): One of 'relu', 'tanh', 'gelu', 'leaky_relu'.
                Unrecognized names silently fall back to ReLU.

        Returns:
            nn.Module: The resolved activation layer.
        """
        return {
            "relu": nn.ReLU(),
            "tanh": nn.Tanh(),
            "gelu": nn.GELU(),
            "leaky_relu": nn.LeakyReLU()
        }.get(name.lower(), nn.ReLU())

    def _data_pre_process(self, x):
        """Embed every input feature and concatenate them into one tensor.

        Args:
            x (dict[str, torch.Tensor]): Batch dict with keys 'latitude',
                'longitude', 'building_type', 'load', 'day_of_year',
                'day_of_week', 'hour_of_day'.

        Returns:
            torch.Tensor: shape (batch, seq_len, 256) -- the concatenated
                embeddings (32*6 + 64 = 256 channels), ready for a
                subclass's own layers.
        """
        lat = self.embeddings['lat'](x['latitude'])
        lon = self.embeddings['lon'](x['longitude'])
        btype = self.embeddings['building'](x['building_type'].squeeze(-1))
        load = self.embeddings['power'](x['load'])
        day_of_year = self.embeddings['day_of_year'](x['day_of_year'])
        day_of_week = self.embeddings['day_of_week'](x['day_of_week'])
        hour_of_day = self.embeddings['hour_of_day'](x['hour_of_day'])
        return torch.cat([lat, lon, btype, day_of_year, day_of_week, hour_of_day, load], dim=2)


class NN(Model):
    """
    Feedforward baseline. Flattens the entire embedded context window into
    one vector and maps it straight to the prediction window through a
    single hidden layer. Ignores sequence order entirely (unlike
    RNN/LSTM/GRU) -- useful as a lower bound for judging whether the
    recurrent models are actually learning temporal structure.
    """
    def __init__(self, activation):
        super().__init__(activation)
        self.model = self._build_model()

    def _build_model(self):
        """Build a 2-layer MLP: input_dim -> 128 -> pred_len.

        Returns:
            nn.Sequential
        """
        input_dim = self.context_len * 256
        return nn.Sequential(
            nn.Linear(input_dim, 128),
            self.activation,
            nn.Linear(128, self.pred_len)
        )

    def forward(self, x):
        """Args:
            x (dict[str, torch.Tensor]): Batch dict, see Model._data_pre_process.

        Returns:
            torch.Tensor: shape (batch, pred_len, 1) -- the forecast.
        """
        ts_embed = self._data_pre_process(x)
        x_flat = ts_embed[:, :self.context_len, :].reshape(x['load'].shape[0], -1)
        return self.model(x_flat).unsqueeze(-1)


class RNN(Model):
    """
    Two-layer vanilla RNN. Encodes the full embedded (context+pred)
    sequence, takes the final hidden state, and maps it to the pred_len
    forecast. Simpler and more prone to vanishing gradients than LSTM/GRU
    on long sequences -- a useful comparison point for showing *why* gated
    recurrent units exist.
    """
    def __init__(self, activation="relu"):
        super().__init__(activation)
        self.rnn1, self.rnn2, self.output_layer = self._build_model()

    def _build_model(self):
        """Build a 2-layer stacked RNN (256->128->128) plus an output projection.

        Returns:
            tuple[nn.RNN, nn.RNN, nn.Linear]
        """
        rnn1 = nn.RNN(256, 128, batch_first=True)
        rnn2 = nn.RNN(128, 128, batch_first=True)
        output_layer = nn.Linear(128, self.pred_len)
        return rnn1, rnn2, output_layer

    def forward(self, x):
        """Args:
            x (dict[str, torch.Tensor]): Batch dict, see Model._data_pre_process.

        Returns:
            torch.Tensor: shape (batch, pred_len, 1) -- the forecast.
        """
        ts_embed = self._data_pre_process(x)
        out1, _ = self.rnn1(ts_embed)
        out2, _ = self.rnn2(out1)
        last_hidden = self.activation(out2[:, -1, :])
        return self.output_layer(last_hidden).unsqueeze(-1)


class LSTM(Model):
    """
    Two-layer LSTM. Same shape as RNN, but its gating (input/forget/output
    gates) lets it retain information over much longer sequences without
    vanishing gradients -- typically the strongest of the four built-in
    architectures on a 168-hour context window.
    """
    def __init__(self, activation="relu"):
        super().__init__(activation)
        self.lstm1, self.lstm2, self.output_layer = self._build_model()

    def _build_model(self):
        """Build a 2-layer stacked LSTM (256->128->128) plus an output projection.

        Returns:
            tuple[nn.LSTM, nn.LSTM, nn.Linear]
        """
        lstm1 = nn.LSTM(256, 128, batch_first=True)
        lstm2 = nn.LSTM(128, 128, batch_first=True)
        output_layer = nn.Linear(128, self.pred_len)
        return lstm1, lstm2, output_layer

    def forward(self, x):
        """Args:
            x (dict[str, torch.Tensor]): Batch dict, see Model._data_pre_process.

        Returns:
            torch.Tensor: shape (batch, pred_len, 1) -- the forecast.
        """
        ts_embed = self._data_pre_process(x)
        out1, _ = self.lstm1(ts_embed)
        out2, _ = self.lstm2(out1)
        last_hidden = self.activation(out2[:, -1, :])
        return self.output_layer(last_hidden).unsqueeze(-1)


class GRU(Model):
    """
    Two-layer GRU. A simpler gating mechanism than LSTM (no separate cell
    state, fewer parameters) that often matches LSTM's accuracy while
    training faster -- a good speed/accuracy tradeoff to compare against
    LSTM directly.
    """
    def __init__(self, activation="relu"):
        super().__init__(activation)
        self.gru1, self.gru2, self.output_layer = self._build_model()

    def _build_model(self):
        """Build a 2-layer stacked GRU (256->128->128) plus an output projection.

        Returns:
            tuple[nn.GRU, nn.GRU, nn.Linear]
        """
        gru1 = nn.GRU(256, 128, batch_first=True)
        gru2 = nn.GRU(128, 128, batch_first=True)
        output_layer = nn.Linear(128, self.pred_len)
        return gru1, gru2, output_layer

    def forward(self, x):
        """Args:
            x (dict[str, torch.Tensor]): Batch dict, see Model._data_pre_process.

        Returns:
            torch.Tensor: shape (batch, pred_len, 1) -- the forecast.
        """
        ts_embed = self._data_pre_process(x)
        out1, _ = self.gru1(ts_embed)
        out2, _ = self.gru2(out1)
        last_hidden = self.activation(out2[:, -1, :])
        return self.output_layer(last_hidden).unsqueeze(-1)


class Trainer:
    """
    Trains one (dataset, model, activation, optimizer, epochs) combination
    end-to-end and writes its results to disk.

    Owns model construction, optimizer selection, the training loop, and
    evaluation -- MAE/RMSE/R² computed in the original, unscaled load
    units via each building's `inverse_transform`. Every combination gets
    its own output directory:
        <cwd>/<dataset>/<model>/<activation>/<optimizer>/epochs-<N>/
    containing train_loss.json, predictions.json, and evaluate_model.json.

    Usage:
        trainer = Trainer(model_name='LSTM', device='cuda:0',
                           scaler_transform='boxcox', dataset_name='ideal',
                           epochs=10, train_buildings=train_buildings,
                           test_buildings=test_buildings, activation='relu',
                           optimizer_name='adam', lr=1e-3)
        train_duration = trainer.train()
        results, mae, rmse, r2 = trainer.evaluate()

    Args:
        model_name (str): One of 'NN', 'RNN', 'LSTM', 'GRU', 'MyNN'.
        device (str): 'cuda:0' or 'cpu'.
        scaler_transform (str): Scaling transform applied to the load
            values; must match what was used to build train/test_buildings
            (e.g. so `inverse_transform` in evaluate() is correct).
        dataset_name (str): Used only to build the output directory path.
        epochs (int): Number of training epochs.
        train_buildings (list[tuple[str, TorchBuildingDataset]]): Training
            split, as returned by DataHandler.load_dataset.
        test_buildings (list[tuple[str, TorchBuildingDataset]]): Held-out
            evaluation split.
        activation (str): Passed through to the model constructor.
        optimizer_name (str): One of 'adam', 'sgd', 'adamw'.
        lr (float): Learning rate.
    """
    def __init__(self, model_name, device, scaler_transform, dataset_name, epochs,
                 train_buildings, test_buildings, activation='relu',
                 optimizer_name='adam', lr=1e-3):
        self.model_name = model_name
        self.device = device
        self.scaler_transform = scaler_transform
        self.dataset_name = dataset_name
        self.epochs = epochs
        self.train_buildings = train_buildings
        self.test_buildings = test_buildings
        self.activation = activation
        self.optimizer_name = optimizer_name
        self.lr = lr
        self.model = self._load_model()
        self.optimizer = self._get_optimizer()
        self.loss_fn = nn.MSELoss()
        self.handler = DataHandler(batch_size=32)
        self.path = os.path.join(os.getcwd(), dataset_name, model_name, activation,
                                  optimizer_name, f'epochs-{epochs}')
        os.makedirs(self.path, exist_ok=True)

    def _load_model(self):
        """Instantiate the requested architecture by name.

        Returns:
            Model: The model, moved to self.device.

        Raises:
            KeyError: If self.model_name isn't one of NN/RNN/LSTM/GRU/MyNN.
        """
        model_map = {
            'NN': NN,
            'RNN': RNN,
            'LSTM': LSTM,
            'GRU': GRU,
            'MyNN': MyNN
        }
        return model_map[self.model_name](activation=self.activation).to(self.device)

    def _get_optimizer(self):
        """Build the optimizer for self.model's parameters.

        Returns:
            torch.optim.Optimizer: Falls back to Adam if optimizer_name
                isn't recognized.
        """
        opt_map = {
            'adam': torch.optim.Adam,
            'sgd': torch.optim.SGD,
            'adamw': torch.optim.AdamW
        }
        optimizer_cls = opt_map.get(self.optimizer_name.lower(), torch.optim.Adam)
        return optimizer_cls(self.model.parameters(), lr=self.lr)

    def train(self):
        """Run the full training loop for self.epochs epochs.

        Iterates every building's DataLoader each epoch, computing MSE
        loss between the model's forecast and the ground-truth load
        beyond context_len. Writes per-epoch loss plus total wall-clock
        training time to train_loss.json in self.path.

        Returns:
            float: Total training duration in seconds.
        """
        self.model.train()
        log = []
        start_time = time.time()
        for epoch in range(self.epochs):
            total_loss = 0.0
            for building_id, building_dataset in self.train_buildings:
                dataloader = self.handler.create_dataloader(building_dataset)
                for batch in dataloader:
                    for key, value in batch.items():
                        batch[key] = value.to(self.device)
                    self.optimizer.zero_grad()
                    predictions = self.model(batch)
                    targets = batch['load'][:, self.model.context_len:, 0]
                    loss = self.loss_fn(predictions[:, :, 0], targets)
                    loss.backward()
                    self.optimizer.step()
                    total_loss += loss.item()
            print(f"[{self.model_name}] Epoch {epoch + 1}: Loss = {total_loss:.4f}", flush=True)
            log.append({"epoch": epoch + 1, "loss": total_loss})
        train_duration = time.time() - start_time
        with open(os.path.join(self.path, "train_loss.json"), "w") as f:
            json.dump({"train_loss": log, "train_duration": train_duration}, f, indent=2)
        return train_duration

    def evaluate(self):
        """Evaluate the trained model on self.test_buildings.

        Runs inference with no gradient tracking, inverse-transforms
        predictions/targets/loads back to real kWh units per building
        (undoing whatever scaler_transform was applied at load time), and
        averages MAE/RMSE/R² across buildings. Writes raw predictions to
        predictions.json and the averaged metrics to evaluate_model.json.

        Returns:
            tuple[dict, float, float, float]: (per-building results dict
                with 'load'/'predictions'/'targets' lists, mean MAE, mean
                RMSE, mean R²).
        """
        self.model.eval()
        results = {}
        mae_total = 0.0
        rmse_total = 0.0
        r2_total = 0.0
        count = 0
        for building_id, building_dataset in self.test_buildings:
            inverse_transform = building_dataset.datasets[0].load_transform.undo_transform
            dataloader = self.handler.create_dataloader(building_dataset)

            target_list = []
            prediction_list = []
            load_list = []

            with torch.no_grad():
                for batch in dataloader:
                    for key, value in batch.items():
                        batch[key] = value.to(self.device)

                    predictions = self.model(batch)
                    targets = batch['load'][:, self.model.context_len:]
                    loads = batch['load'][:, :self.model.context_len]

                    targets = inverse_transform(targets)
                    predictions = inverse_transform(predictions)
                    loads = inverse_transform(loads)

                    prediction_list.append(predictions.detach().cpu())
                    target_list.append(targets.detach().cpu())
                    load_list.append(loads.detach().cpu())

            predictions_all = torch.cat(prediction_list)
            targets_all = torch.cat(target_list)
            load_all = torch.cat(load_list)

            mae = torch.abs(predictions_all - targets_all).mean().item()
            rmse = torch.sqrt(((predictions_all - targets_all) ** 2).mean()).item()
            r2 = 1 - (((predictions_all - targets_all) ** 2).sum() /
                      ((targets_all - targets_all.mean()) ** 2).sum()).item()
            mae_total += mae
            rmse_total += rmse
            r2_total += r2
            count += 1
            results[building_id] = {
                "load": load_all.tolist(),
                "predictions": predictions_all.tolist(),
                "targets": targets_all.tolist()
            }
        with open(os.path.join(self.path, "predictions.json"), "w") as f:
            json.dump(results, f, indent=2)
        eval_metrics = {
            "mae": mae_total / count,
            "rmse": rmse_total / count,
            "r2": r2_total / count}
        with open(os.path.join(self.path, "evaluate_model.json"), "w") as f:
            json.dump(eval_metrics, f, indent=2)
        return results, eval_metrics["mae"], eval_metrics["rmse"], eval_metrics["r2"]

# ------------------- #
# --- Do Not Edit --- #
# ------------------- #
print("Finished!")

/global/common/software/m4388/lgupta/conda/buildingsEnv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Finished!


In [ ]:
!pwd

## Task 1

We want to first test our training loop and make sure it works. Let's select a few combos of datasets, models, activations, optimizers and epochs, and make sure that the training loop below runs successfully. 

In [2]:
# ------------------- #
# ------ Edit ------- #
# ------------------- #
# TODO: Explore at least a few different combinations to ensure the loop runs. Continue to experiment to see how long the training(s) take.
dataset_names = 
model_classes = 
activations = 
optimizers = 
epoch_options = 

# ------------------- #
# ------ Edit ------- #
# ------------------- #


# ------------------- #
# --- Do Not Edit --- #
# ------------------- #

class MyNN(Model):
     def __init__(self):
         pass

os.environ["REPO_PATH"] = "/pscratch/sd/l/lgupta/buildingsBench/"
os.environ["BUILDINGS_BENCH"] = "/global/cfs/cdirs/m4388/2025_Bootcamp/Project4/Dataset"
os.environ["TRANSFORM_PATH"] = "/global/cfs/cdirs/m4388/2025_Bootcamp/Project4/Dataset/metadata/transforms"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

for dataset_name in dataset_names:
    print(f"\n=== Dataset: {dataset_name} ===")
    handler = DataHandler(batch_size=32)
    all_buildings = handler.load_dataset(dataset_name, scaler_transform="boxcox")
    train_buildings = all_buildings[:int(0.8 * len(all_buildings))]
    test_buildings = all_buildings[int(0.8 * len(all_buildings)):]
    for model_class in model_classes:
        for activation in activations:
            for optimizer_name in optimizers:
                for epochs in epoch_options:
                    print(f"\n--- Training {model_class} | Activation: {activation} | Optimizer: {optimizer_name} | Epochs: {epochs} ---")
                    trainer = Trainer(
                        model_name=model_class,
                        device=device,
                        dataset_name=dataset_name,
                        epochs=epochs,
                        train_buildings=train_buildings,
                        test_buildings=test_buildings,
                        scaler_transform="boxcox",
                        activation=activation,
                        optimizer_name=optimizer_name,
                        lr=1e-3)
                    train_duration = trainer.train()
                    results, mae, rmse, r2 = trainer.evaluate()
                    print(f"[{model_class}] MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
                    print(f"Training Time: {train_duration:.2f} seconds")
                    
# ------------------- #
# --- Do Not Edit --- #
# ------------------- #


=== Dataset: ideal ===

--- Training NN | Activation: relu | Optimizer: adam | Epochs: 1 ---
[NN] Epoch 1: Loss = 3030.0398
[NN] MAE: 0.5158, RMSE: 0.5770, R²: -3.6049
Training Time: 47.44 seconds


## Task 2

Try making your own neural network! Can you design your own model and get it to perform better than the given models? Use the PyTorch documentation to figure out how to write your own model.

In [ ]:
# ------------------- #
# ------ Edit ------- #
# ------------------- #

class MyNN(Model):
    def __init__(self, activation):
        super().__init__(activation)
        self.model = self._build_model()

    def _build_model(self):
        input_dim = self.context_len * 256
        return nn.Sequential(
            nn.Linear(input_dim, 256),     # input layer
            self.activation,
            nn.Linear(256, 128),           # hidden layer 1
            self.activation,
            nn.Linear(128, 64),            # hidden layer 2
            self.activation,
            nn.Linear(64, 32),             # hidden layer 3
            self.activation,
            nn.Linear(32, self.pred_len)   # output layer -- must end at pred_len (24)
        )

    def forward(self, x):
        ts_embed = self._data_pre_process(x)
        x_flat = ts_embed[:, :self.context_len, :].reshape(x['load'].shape[0], -1)
        return self.model(x_flat).unsqueeze(-1)


# TODO: Set up the training combinations to test that your model can train successfully.
dataset_names = 
model_classes = 
activations = 
optimizers = 
epoch_options = 


# ------------------- #
# ------ Edit ------- #
# ------------------- #

# ------------------- #
# --- Do Not Edit --- #
# ------------------- #

os.environ["REPO_PATH"] = "/pscratch/sd/l/lgupta/buildingsBench/"
os.environ["BUILDINGS_BENCH"] = "/global/cfs/cdirs/m4388/2025_Bootcamp/Project4/Dataset"
os.environ["TRANSFORM_PATH"] = "/global/cfs/cdirs/m4388/2025_Bootcamp/Project4/Dataset/metadata/transforms"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

for dataset_name in dataset_names:
    print(f"\n=== Dataset: {dataset_name} ===")
    handler = DataHandler(batch_size=32)
    all_buildings = handler.load_dataset(dataset_name, scaler_transform="boxcox")
    train_buildings = all_buildings[:int(0.8 * len(all_buildings))]
    test_buildings = all_buildings[int(0.8 * len(all_buildings)):]
    for model_class in model_classes:
        for activation in activations:
            for optimizer_name in optimizers:
                for epochs in epoch_options:
                    print(f"\n--- Training {model_class} | Activation: {activation} | Optimizer: {optimizer_name} | Epochs: {epochs} ---")
                    trainer = Trainer(
                        model_name=model_class,
                        device=device,
                        dataset_name=dataset_name,
                        epochs=epochs,
                        train_buildings=train_buildings,
                        test_buildings=test_buildings,
                        scaler_transform="boxcox",
                        activation=activation,
                        optimizer_name=optimizer_name,
                        lr=1e-3)
                    train_duration = trainer.train()
                    results, mae, rmse, r2 = trainer.evaluate()
                    print(f"[{model_class}] MAE: {mae:.4f}, RMSE: {rmse:.4f}, R²: {r2:.4f}")
                    print(f"Training Time: {train_duration:.2f} seconds")

# ------------------- #
# --- Do Not Edit --- #
# ------------------- #

## Next Step

Running all of the combinations in a notebook is not efficient. We want to parallelize! Wait until instructed to learn how to submit a parallelized version of this script. After we run several combinations we can move on to:

`/BuildingsBenchTutorial/Tutorials/Final-Project-Modules/Select-Model.ipynb`